# Q-Neural-Dynamics: Quantum-Inspired Neural Framework
### CPU-Optimized Execution Notebook for Biological & Neural Time-Series

In [ ]:
# Step 0: Force reinstall clean and compatible versions for Colab Python environment
print("Cleaning and installing compatible scientific packages...")
!pip uninstall -y numpy scikit-learn pandas > /dev/null 2>&1
!pip install -q "numpy>=1.26.0,<2.0" pandas scikit-learn torch matplotlib

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Enforcing strict CPU device allocation
device = torch.device("cpu")
print(f"Active computational device: {device}")

In [ ]:
# Step 1: Define Quantum-Inspired Orthogonal Reservoir Layer
class QuantumInspiredReservoir(nn.Module):
    def __init__(self, input_dim, reservoir_dim):
        super(QuantumInspiredReservoir, self).__init__()
        self.input_dim = input_dim
        self.reservoir_dim = reservoir_dim
        
        # QR decomposition for orthogonal transformation (unitary-inspired weights)
        raw_weights = torch.randn(reservoir_dim, input_dim)
        Q, _ = torch.linalg.qr(raw_weights)
        self.W_in = nn.Parameter(Q, requires_grad=False)
        
        recurrent_raw = torch.randn(reservoir_dim, reservoir_dim)
        Q_rec, _ = torch.linalg.qr(recurrent_raw)
        self.W_rec = nn.Parameter(Q_rec * 0.99, requires_grad=False)
        
    def forward(self, x):
        batch_size, seq_len, _ = x.size()
        h = torch.zeros(batch_size, self.reservoir_dim, device=x.device)
        
        states = []
        for t in range(seq_len):
            xt = x[:, t, :]
            h = torch.tanh(torch.matmul(xt, self.W_in.T) + torch.matmul(h, self.W_rec.T))
            states.append(h.unsqueeze(1))
            
        return torch.cat(states, dim=1)

# Step 2: Hybrid Classifier Model
class QNeuralClassifier(nn.Module):
    def __init__(self, input_dim, reservoir_dim, num_classes):
        super(QNeuralClassifier, self).__init__()
        self.reservoir = QuantumInspiredReservoir(input_dim, reservoir_dim)
        self.classifier = nn.Sequential(
            nn.Linear(reservoir_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )
        
    def forward(self, x):
        res_states = self.reservoir(x)
        repr_features = torch.mean(res_states, dim=1)
        out = self.classifier(repr_features)
        return out

print("Model architecture initialized successfully on CPU.")

In [ ]:
# Step 3: Dataset Loading & Benchmark Generation
def load_and_preprocess_data(file_path, target_column, seq_len=30):
    df = pd.read_csv(file_path)
    y = df[target_column].values
    X = df.drop(columns=[target_column]).values
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    num_samples = len(X_scaled) - seq_len
    X_sequences, y_sequences = [], []
    for i in range(num_samples):
        X_sequences.append(X_scaled[i : i + seq_len])
        y_sequences.append(y[i + seq_len])
    return np.array(X_sequences, dtype=np.float32), np.array(y_sequences, dtype=np.int64)

# Generate biological benchmark data
np.random.seed(42)
sample_data = np.random.randn(1000, 11)
cols = [f"feature_{i}" for i in range(10)] + ["label"]
pd.DataFrame(sample_data, columns=cols).to_csv("biological_data.csv", index=False)
print("Benchmark dataset generated successfully.")

In [ ]:
# Step 4: Training Pipeline on CPU
X_data, y_data = load_and_preprocess_data('biological_data.csv', 'label', seq_len=20)
X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.2, random_state=42)

X_train_t = torch.tensor(X_train, device=device)
y_train_t = torch.tensor(y_train, device=device)
X_test_t = torch.tensor(X_test, device=device)
y_test_t = torch.tensor(y_test, device=device)

model = QNeuralClassifier(input_dim=X_train.shape[2], reservoir_dim=64, num_classes=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

print("\n--- Training on CPU ---")
model.train()
for epoch in range(10):
    optimizer.zero_grad()
    loss = criterion(model(X_train_t), y_train_t)
    loss.backward()
    optimizer.step()
    print(f"Epoch [{epoch+1}/10] | Loss: {loss.item():.4f}")

model.eval()
with torch.no_grad():
    acc = (torch.max(model(X_test_t), 1)[1] == y_test_t).sum().item() / len(y_test_t)
print(f"\nTraining complete! Test Accuracy: {acc * 100:.2f}%")